# Pest Detection — MobileNetV2 Training
**Before running:** Make sure GPU is enabled → Runtime > Change runtime type > T4 GPU

In [ ]:
# ── STEP 1: Check GPU ────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── STEP 2: Mount Google Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── STEP 3: Copy dataset from Drive to Colab local storage ───────────────
# Much faster to read locally than from Drive during training
import shutil, os
from pathlib import Path

DRIVE_DATASET = '/content/drive/MyDrive/prepared_pest_dataset'
LOCAL_DATASET = '/content/prepared_pest_dataset'

if not Path(LOCAL_DATASET).exists():
    print('Copying dataset to local storage...')
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
    print('Done.')
else:
    print('Dataset already copied.')

# Show class-wise counts
from collections import Counter
counts = Counter()
for cls_dir in sorted(Path(LOCAL_DATASET).iterdir()):
    if cls_dir.is_dir():
        n = len(list(cls_dir.iterdir()))
        counts[cls_dir.name] = n
        print(f'  {cls_dir.name:<20} {n:>5} images')
print(f"  {'TOTAL':<20} {sum(counts.values()):>5} images")

In [ ]:
# ── STEP 4: Install dependencies ─────────────────────────────────────────
!pip install -q scikit-learn matplotlib seaborn

In [ ]:
# ── STEP 5: Config ───────────────────────────────────────────────────────
IMG_SIZE        = 224
BATCH_SIZE      = 32
EPOCHS_FROZEN   = 10
EPOCHS_FINETUNE = 20
UNFREEZE_LAYERS = 30
SEED            = 42

DATASET_DIR = '/content/prepared_pest_dataset'
OUTPUT_DIR  = '/content/pest_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

tf.random.set_seed(SEED)
import numpy as np
np.random.seed(SEED)

print('Config set.')

In [ ]:
# ── STEP 6: Build tf.data pipelines ─────────────────────────────────────
from tensorflow.keras import layers

SIZE = (IMG_SIZE, IMG_SIZE)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.30,
    subset='training',
    label_mode='categorical',
)

remaining = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.30,
    subset='validation',
    label_mode='categorical',
)

class_names = train_ds.class_names
n_classes   = len(class_names)
print(f'{n_classes} classes: {class_names}')

# Split remaining 30% into val (15%) and test (15%)
half     = len(remaining) // 2
val_ds   = remaining.take(half)
test_ds  = remaining.skip(half)

# Augmentation + normalisation
augment   = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.12),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
])
normalise = layers.Rescaling(1.0 / 255)
AUTOTUNE  = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .map(lambda x, y: (normalise(x), y), num_parallel_calls=AUTOTUNE)
    .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds
    .map(lambda x, y: (normalise(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds
    .map(lambda x, y: (normalise(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

print('Pipelines ready.')

In [ ]:
# ── STEP 7: Save class names ─────────────────────────────────────────────
import json
with open(f'{OUTPUT_DIR}/pest_class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
print('Saved pest_class_names.json')

In [ ]:
# ── STEP 8: Build model ──────────────────────────────────────────────────
from tensorflow.keras import models, optimizers
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base.trainable = False

inputs  = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(n_classes, activation='softmax')(x)

model = models.Model(inputs, outputs, name='PestNet')
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# ── STEP 9: Phase 1 — Train head only ───────────────────────────────────
from tensorflow.keras import callbacks

cbs_p1 = [
    callbacks.ModelCheckpoint(
        f'{OUTPUT_DIR}/best_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1),
]

print('='*50)
print('PHASE 1 — Training head (base frozen)')
print('='*50)
h1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    callbacks=cbs_p1,
)

In [ ]:
# ── STEP 10: Phase 2 — Fine-tune last 30 layers ──────────────────────────
base.trainable = True
for layer in base.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

cbs_p2 = [
    callbacks.ModelCheckpoint(
        f'{OUTPUT_DIR}/best_phase2.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1),
]

print('='*50)
print(f'PHASE 2 — Fine-tuning last {UNFREEZE_LAYERS} layers')
print('='*50)
h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=cbs_p2,
)

In [ ]:
# ── STEP 11: Save final model ────────────────────────────────────────────
model.save(f'{OUTPUT_DIR}/pest_model.keras')
print('Saved pest_model.keras')

In [ ]:
# ── STEP 12: Evaluate on test set ────────────────────────────────────────
print('\n── Test Set Evaluation ─────────────────────────────')
loss, acc = model.evaluate(test_ds, verbose=1)
print(f'  Test Accuracy : {acc:.4f}')
print(f'  Test Loss     : {loss:.4f}')

In [ ]:
# ── STEP 13: Training history plot ───────────────────────────────────────
import matplotlib.pyplot as plt

acc   = h1.history['accuracy']     + h2.history['accuracy']
val   = h1.history['val_accuracy'] + h2.history['val_accuracy']
loss  = h1.history['loss']         + h2.history['loss']
vloss = h1.history['val_loss']     + h2.history['val_loss']
ep    = range(1, len(acc) + 1)
split = len(h1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(ep, acc,  label='Train Acc')
ax1.plot(ep, val,  label='Val Acc')
ax1.axvline(split, color='gray', linestyle='--', label='Unfreeze')
ax1.set_title('Accuracy'); ax1.legend()

ax2.plot(ep, loss,  label='Train Loss')
ax2.plot(ep, vloss, label='Val Loss')
ax2.axvline(split, color='gray', linestyle='--', label='Unfreeze')
ax2.set_title('Loss'); ax2.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_history.png', dpi=150)
plt.show()
print('Saved training_history.png')

In [ ]:
# ── STEP 14: Confusion matrix ────────────────────────────────────────────
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion_matrix.png')

print('\n── Classification Report ───────────────────────────')
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ── STEP 15: Export TFLite ───────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = f'{OUTPUT_DIR}/pest_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = Path(tflite_path).stat().st_size / (1024 * 1024)
print(f'Saved pest_model.tflite ({size_mb:.1f} MB)')

In [ ]:
# ── STEP 16: Copy all outputs back to Google Drive ───────────────────────
DRIVE_OUTPUT = '/content/drive/MyDrive/pest_model_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for filename in [
    'pest_model.keras',
    'pest_model.tflite',
    'pest_class_names.json',
    'training_history.png',
    'confusion_matrix.png',
]:
    src = f'{OUTPUT_DIR}/{filename}'
    if Path(src).exists():
        shutil.copy(src, f'{DRIVE_OUTPUT}/{filename}')
        print(f'  Copied {filename} → Drive')

print('\n✓ All outputs saved to Google Drive/pest_model_outputs/')